# Phase 3 — Azure Predictive Maintenance Baseline Modeling & Evaluation

This notebook provides the official, reproducible baseline modeling, checkpointing, thresholding, and evaluation workflow for the Microsoft Azure Predictive Maintenance (PdM) track in **FedMaint-AI**.

## 1. Scope and Architecture Overview

Phase 3 establishes rigorous single-client (local-only) baselines and an un-privatized centralized reference across three model architectures:

1. **AzureGRU Backbone (`AzureGRU`):**
   - Single-layer GRU (`input_size=4`, `hidden_size=32`, `num_layers=1`, `dropout=0.0`, `batch_first=True`).
   - Input sequence: $[N, 48, 4]$ normalized sensor readings (`volt`, `rotate`, `pressure`, `vibration`) from the past-only observation window $[t-48\text{h}, t)$.
   - Summary head: Concatenates the final hidden state ($32$-dim) with $8$ historical summary features ($8$-dim: deterministic `signed_log1p` of past median and IQR for each sensor) into a $40$-dim representation before the linear classification head (`[40 -> 1]`).
   - Objective: Unweighted `BCEWithLogitsLoss`.

2. **Tabular Logistic Regression Baseline (`AzureLogisticRegression`):**
   - Scikit-learn `LogisticRegression` (`solver='lbfgs'`, $C=1.0$, `max_iter=1000`) on $20$ causal tabular features per window (median, IQR, last raw reading, last-minus-first raw delta, and population standard deviation across the 48 raw readings for all 4 sensors).
   - Scaler: Strict train-fitted `TabularStandardScaler`. Scaler fitting on validation, test, or held-out splits is blocked by `HoldoutLeakageError`.

3. **Tabular MLP Baseline (`AzureMLP`):**
   - Two-layer neural network ($[20 \to 32 \to \text{ReLU} \to 1]$ logit) trained on the same $20$ causal tabular features with the same neural budget as the GRU.
   - Scaler: Strict train-fitted `TabularStandardScaler`.

## 2. Regimes, Client Boundaries, and Held-Out Factory Isolation

- **Participating Federated Clients:**
  - `client_model2` (17 machines, model2): allowed splits are `train`, `val`, `test`.
  - `client_model3` (35 machines, model3): allowed splits are `train`, `val`, `test`.
  - `client_model4` (32 machines, model4): allowed splits are `train`, `val`, `test`.
- **Strict Held-Out Client (`heldout_model1`):**
  - All 16 machines belonging to `model1` are strictly withheld from training and validation.
  - Contributes zero training updates, validation checks, hyperparameter tuning, or threshold selection.
  - Evaluated exclusively zero-shot on its `test` split using frozen models and frozen validation thresholds.
  - Attempting to load or cache `heldout_model1` for `train` or `val` raises `HoldoutLeakageError`.
- **Training Regimes:**
  - **Local Regime:** Trains 3 independent models (one per participating client). Checkpoint selected on own-client validation AP; threshold selected on own-client validation F1; evaluated on own test split and zero-shot on held-out `model1`.
  - **Centralized Regime:** Pools participating client training partitions (`client_model2`, `client_model3`, `client_model4`). Checkpoint selected on unweighted macro validation AP across participating clients; threshold selected on pooled participating validation F1; evaluated on per-client test, pooled test, macro test, and zero-shot on held-out `model1`. Serves as an un-privatized upper reference.

## 3. Checkpointing, Threshold Selection & Dual Evaluation

- **Validation AP Checkpointing:** Model weights are saved strictly on improvements in validation Average Precision (AP / PR-AUC), preserving the earliest epoch on ties.
- **Validation F1 Threshold Selection:** The decision threshold is chosen by sweeping validation probabilities in $O(N \log N)$ to maximize F1 score. On ties, the highest threshold is chosen deterministically.
- **Dual Evaluation Framework:**
  - **Window-Level:** Evaluates window predictions against binary labels ($y \in \{0, 1\}$) at the frozen threshold: Average Precision, ROC-AUC, F1, Precision, Recall, and confusion matrix.
  - **Event-Level (Operational):** Evaluates failure episode detection. Generates positive alerts chronologically with a 24-hour alert cooldown (suppressing repeat alerts for 24h after an alert). Matches alerts to the earliest future failure episode occurring within $(t, t+24\text{h}]$. Reports Event Recall, False Alert Rate (FAR per 100 eligible machine-days), and mean lead time in hours.
  - **Machine-Cluster Bootstrap:** Resamples entire machines with replacement to calculate 95% percentile confidence intervals for all primary metrics.

## 4. Execution Modes and Safety Guardrails

- **Smoke Mode (Default):** 1 seed (`17`), 2 epochs, bounded subset ($\le 60$ samples per partition), 30 bootstrap draws. Purpose: rapid plumbing and pipeline verification. Explicitly marked non-representative.
- **Full Mode:** 3 approved seeds (`17, 23, 41`), 25 epochs (patience 5), full partitions, 1000 bootstrap draws. Purpose: representative scientific evaluation. Requires intentional parameter change to avoid accidental long runs.
- **Data-Free Safety:** When raw CSVs are not present (such as in this staging repository), the notebook demonstrates architecture contracts and safely skips execution without errors.

In [ ]:
import os
import sys
from pathlib import Path

# Ensure project root and src/ are in sys.path
current_path = Path.cwd().resolve()
if (current_path / "src").is_dir():
    repo_root = current_path
elif (current_path.parent / "src").is_dir():
    repo_root = current_path.parent
else:
    repo_root = current_path

src_path = str(repo_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
repo_path = str(repo_root)
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

from fedmaint.experiments.phase3_azure import (
    DEFAULT_CONFIG_PATH,
    DEFAULT_OUTPUT_DIR,
    Phase3AzureConfig,
    Phase3ExperimentReport,
    SingleRunResult,
    load_phase3_azure_config,
    run_phase3_experiment,
)
from fedmaint.data.splits import HoldoutLeakageError
from fedmaint.models.gru import AzureGRU, is_torch_available
from fedmaint.models.logistic import AzureLogisticRegression, is_sklearn_available
from fedmaint.models.mlp import AzureMLP
from fedmaint.models.scaler import TabularStandardScaler
from scripts.run_phase3_azure import format_summary_table

print(f"FedMaint-AI loaded successfully from: {src_path}")
print(f"PyTorch available:     {is_torch_available()}")
print(f"Scikit-learn available: {is_sklearn_available()}")

## 2. Inspect Frozen Phase 3 Configuration

We load `configs/phase3_azure.json` as the single source of truth for Phase 3 baseline modeling and evaluation.

In [ ]:
config = load_phase3_azure_config(DEFAULT_CONFIG_PATH)
config_hash = config.compute_hash()

print("=" * 80)
print("Phase 3 Azure Configuration Summary")
print("=" * 80)
print(f"Pipeline Name:       {config.pipeline_name}")
print(f"Phase:               {config.phase}")
print(f"Track:               {config.track}")
print(f"Version:             {config.version}")
print(f"SHA-256 Digest:      {config_hash}")
print(f"Description:         {config.description}")
print("-" * 80)
print("Training Hyperparameters (Neural Backbones):")
print(f"  Optimizer:         AdamW (lr={config.lr}, weight_decay={config.weight_decay})")
print(f"  Batch Size:        {config.batch_size}")
print(f"  Loss Function:     BCEWithLogitsLoss (unweighted)")
print(f"  Full Mode:         epochs={config.max_epochs_full}, patience={config.patience_full}, seeds={config.seeds_full}, bootstrap_draws={config.bootstrap_draws_full}")
print(f"  Smoke Mode:        epochs={config.max_epochs_smoke}, patience={config.patience_smoke}, seeds={config.seeds_smoke}, bootstrap_draws={config.bootstrap_draws_smoke}, max_samples={config.smoke_bounded_subset_max_samples}")
print("-" * 80)
print("Model Architectures:")
for model_key, model_info in config.raw_config.get("models", {}).items():
    print(f"  - {model_key:<10}: {model_info.get('name')} | features: {model_info.get('features')}")
print("-" * 80)
print("Participating Clients vs Held-Out Isolation:")
clients_cfg = config.raw_config.get("clients", {})
for c_name, c_info in clients_cfg.get("participating", {}).items():
    print(f"  - {c_name:<16}: model={c_info.get('model')} | allowed={c_info.get('allowed_splits')}")
for h_name, h_info in clients_cfg.get("held_out", {}).items():
    print(f"  - {h_name:<16}: model={h_info.get('model')} | allowed={h_info.get('allowed_splits')} | disallowed={h_info.get('disallowed_splits')}")
    print(f"    Policy: {h_info.get('policy')}")
print("=" * 80)

## 3. Data Directory Resolution & Staging Safety Check

Resolve the directory containing raw Azure CSV tables (`PdM_telemetry.csv`, `PdM_failures.csv`, `PdM_machines.csv`, `PdM_errors.csv`, `PdM_maint.csv`).

If raw CSVs are absent (as in this staging environment), the notebook sets `data_available = False` and will safely skip experiment execution cells without raising errors.

In [ ]:
# Parameterized data directory check
env_data_dir = os.environ.get("AZURE_DATA_DIR")
candidate_dirs = [
    Path(env_data_dir) if env_data_dir else None,
    repo_root / "data" / "Azure Data",
    repo_root.parent / "data" / "Azure Data",
]

data_dir = None
for cand in candidate_dirs:
    if cand is not None and cand.is_dir():
        if (cand / "PdM_telemetry.csv").is_file() and (cand / "PdM_failures.csv").is_file():
            data_dir = cand.resolve()
            break

data_available = data_dir is not None

if data_available:
    print(f"[DATA DETECTED] Validated raw Azure data directory located at: {data_dir}")
else:
    print("[DATA ABSENT] Raw Azure CSV tables not detected in standard search paths.")
    print("             The notebook will run in data-free validation mode.")
    print("             All model contracts and leakage guardrails will be validated using synthetic tensors.")
    print("             Real experiment execution cells will cleanly skip without errors.")

## 4. Experiment Mode & Parameter Configuration

Configure execution mode, models, regimes, and output directory.

> **Safety Guardrail:** `MODE` defaults to `"smoke"`. Full benchmarking runs 3 seeds across all models and regimes with 1000 bootstrap draws. To prevent accidental long runs on "Run All", setting `MODE = "full"` requires explicitly setting `CONFIRM_FULL_RUN = True`.

In [ ]:
# ----------------------------------------------------------------------------
# Experiment Mode Selection
# Default is 'smoke': 1 seed (17), 2 epochs, bounded subset (<=60 samples), 30 bootstrap draws.
# ----------------------------------------------------------------------------
MODE = "smoke"  # 'smoke' or 'full'

# Intentional confirmation flag required for full mode:
CONFIRM_FULL_RUN = False

# Output directory (configurable)
OUTPUT_DIR = repo_root / "outputs" / "phase3"
CACHE_DIR = OUTPUT_DIR / "cache"

# Models to execute: 'all', or a list subset of ['gru', 'logistic', 'mlp']
MODELS_TO_RUN = ["all"]

# Regimes to execute: 'all', or a list subset of ['local', 'central']
REGIMES_TO_RUN = ["all"]

# Optional override for random seeds (None uses config defaults for mode)
SEEDS_OVERRIDE = None

# Force rebuild of memmapped model cache
FORCE_REBUILD_CACHE = False

# Verify full run safety guardrail
if MODE == "full" and not CONFIRM_FULL_RUN:
    raise ValueError(
        "Full mode requested without explicit confirmation!\n"
        "Running in 'full' mode trains all models across seeds [17, 23, 41] for 25 epochs "
        "with 1000 bootstrap draws.\n"
        "To proceed, set CONFIRM_FULL_RUN = True in this cell."
    )

print("Experiment Parameters Configured:")
print(f"  Mode:             {MODE.upper()}")
if MODE == "smoke":
    print("  Notice:           SMOKE mode: plumbing verification only; NOT representative.")
print(f"  Output Directory: {OUTPUT_DIR}")
print(f"  Cache Directory:  {CACHE_DIR}")
print(f"  Models:           {MODELS_TO_RUN}")
print(f"  Regimes:          {REGIMES_TO_RUN}")
print(f"  Seeds Override:   {SEEDS_OVERRIDE}")

## 5. Architectural Contract & Leakage Guardrail Verification

Even without raw CSV files, we verify the mathematical shapes, tensor contracts, and leakage guardrails of all Phase 3 models in memory:

1. **`AzureGRU` Forward Pass:** Input $[B, 48, 4]$ and summaries $[B, 8] \to$ logits $[B, 1]$.
2. **`AzureMLP` Forward Pass:** Input $[B, 20] \to$ logits $[B, 1]$.
3. **`TabularStandardScaler` Guardrail:** Fitted strictly on `split_name='train'`; rejects `val` or `test` with `HoldoutLeakageError`.
4. **`AzureLogisticRegression` Guardrail:** Fits only on `split_name='train'`.

In [ ]:
import numpy as np

print("=" * 80)
print("Verifying Phase 3 Model Contracts & Guardrails (Data-Free)")
print("=" * 80)
print("")
# 1. AzureGRU Tensor Contract
if is_torch_available():
    import torch
    gru_model = AzureGRU(input_size=4, hidden_size=32, num_summaries=8, output_size=1)
    dummy_seq = torch.randn(4, 48, 4)
    dummy_sum = torch.randn(4, 8)
    with torch.no_grad():
        out_logits = gru_model(dummy_seq, dummy_sum)
    assert out_logits.shape == (4, 1), f"Unexpected GRU output shape: {out_logits.shape}"
    print(f"[PASS] AzureGRU contract verified: seq [4, 48, 4] + sum [4, 8] -> logits {list(out_logits.shape)}")
else:
    print("[SKIP] PyTorch not available for AzureGRU test.")

# 2. AzureMLP Tensor Contract
if is_torch_available():
    mlp_model = AzureMLP(input_dim=20, hidden_dim=32, output_dim=1)
    dummy_tab = torch.randn(4, 20)
    with torch.no_grad():
        out_mlp = mlp_model(dummy_tab)
    assert out_mlp.shape == (4, 1), f"Unexpected MLP output shape: {out_mlp.shape}"
    print(f"[PASS] AzureMLP contract verified: tabular [4, 20] -> logits {list(out_mlp.shape)}")
else:
    print("[SKIP] PyTorch not available for AzureMLP test.")

# 3. TabularStandardScaler Leakage Guardrail
scaler = TabularStandardScaler()
dummy_train_X = [[1.0] * 20, [2.0] * 20, [3.0] * 20]

# Train split is permitted
scaler.fit(dummy_train_X, split_name="train")
assert scaler.is_fitted
print("[PASS] TabularStandardScaler accepts split_name='train'")

# Validation or Test split must raise HoldoutLeakageError
for disallowed_split in ["val", "test", "heldout_model1", None]:
    try:
        scaler.fit(dummy_train_X, split_name=disallowed_split)
        raise AssertionError(f"Security failure: scaler fitted on disallowed split '{disallowed_split}'!")
    except HoldoutLeakageError:
        pass
print("[PASS] TabularStandardScaler strictly blocks fitting on val, test, or heldout with HoldoutLeakageError")

# 4. AzureLogisticRegression Fitting Guardrail
if is_sklearn_available():
    log_clf = AzureLogisticRegression()
    dummy_y = [0, 1, 0]
    log_clf.fit(dummy_train_X, dummy_y, split_name="train")
    assert log_clf.is_fitted
    try:
        log_clf.fit(dummy_train_X, dummy_y, split_name="val")
        raise AssertionError("Security failure: LogisticRegression allowed fitting on val!")
    except HoldoutLeakageError:
        pass
    print("[PASS] AzureLogisticRegression strictly enforces train-only fitting guardrail")
else:
    print("[SKIP] scikit-learn not available for AzureLogisticRegression test.")

print("=" * 80)

## 6. Execute Baseline Experiments via Real Experiment Orchestration API

This cell calls `run_phase3_experiment` directly from `fedmaint.experiments.phase3_azure`.

The Python API execution is exactly equivalent to invoking the CLI:
```bash
python scripts/run_phase3_azure.py --mode smoke --output-dir outputs/phase3
# or for full scientific benchmarking:
python scripts/run_phase3_azure.py --mode full --output-dir outputs/phase3
```

If raw data is absent, this cell displays an informative skip notice without error.

In [ ]:
report = None

if not data_available:
    print("=" * 80)
    print("NOTICE: Raw Azure CSV tables are not present in this staging workspace.")
    print("Safely skipping experiment execution without errors.")
    print("=" * 80)
    print("When executed in the feature repository with raw CSVs, this cell will:")
    print(f"  1. Load and build memmapped caches under '{CACHE_DIR}'")
    print(f"  2. Train models ({MODELS_TO_RUN}) across regimes ({REGIMES_TO_RUN}) in {MODE.upper()} mode")
    print("  3. Save best checkpoints on validation Average Precision (strict improvement)")
    print("  4. Select decision thresholds maximizing validation F1 score")
    print("  5. Evaluate window metrics, 24h-cooldown event metrics, and 95% bootstrap CIs")
    print("  6. Perform separate zero-shot evaluation on held-out model1 (test split only)")
    print(f"  7. Write structured results to '{OUTPUT_DIR / 'results.json'}'")
    print("\nEquivalent CLI Command:")
    print(f"    python scripts/run_phase3_azure.py --mode {MODE} --output-dir \"{OUTPUT_DIR}\"")
else:
    print("=" * 80)
    print(f"Executing Phase 3 Azure Baselines in {MODE.upper()} Mode...")
    print("=" * 80)
    report = run_phase3_experiment(
        config=DEFAULT_CONFIG_PATH,
        mode=MODE,
        models=MODELS_TO_RUN,
        regimes=REGIMES_TO_RUN,
        seeds=SEEDS_OVERRIDE,
        data_dir=data_dir,
        output_dir=OUTPUT_DIR,
        cache_dir=CACHE_DIR,
        force_rebuild_cache=FORCE_REBUILD_CACHE,
    )
    print("=" * 80)
    print("Experiment Execution Succeeded!")
    print(f"Results JSON: {OUTPUT_DIR / 'results.json'}")
    print(f"Summary File: {OUTPUT_DIR / 'summary.txt'}")
    print("=" * 80)

## 7. Telemetry, Provenance & Hardware Environment Inspection

Inspect the run metadata, SHA-256 configuration hash, Git commit SHA, timestamp, package versions, and hardware device.

In [ ]:
if report is not None:
    print("=" * 80)
    print("Experiment Provenance & Runtime Environment")
    print("=" * 80)
    print(f"Pipeline Name:       {report.pipeline_name}")
    print(f"Phase:               {report.phase}")
    print(f"Execution Mode:      {report.mode}")
    print(f"Is Representative:   {report.is_representative}")
    print(f"Is Plumbing Only:    {report.is_plumbing_only}")
    print(f"Config Hash:         {report.config_hash}")
    print(f"Git Commit SHA:      {report.git_revision or 'Not recorded'}")
    print(f"Created At (UTC):    {report.created_at}")
    print(f"Output Directory:    {report.output_dir}")
    print(f"Models Executed:     {report.models_executed}")
    print(f"Regimes Executed:    {report.regimes_executed}")
    print(f"Seeds Executed:      {report.seeds_executed}")
    print(f"Total Single Runs:   {len(report.results)}")
    
    if report.results:
        sample_run = report.results[0]
        print("\nHardware and Package Versions (from run telemetry):")
        print(f"  Device:             {sample_run.device}")
        for pkg, ver in sample_run.package_versions.items():
            print(f"  {pkg:<18}: {ver}")
        if sample_run.environment_info:
            print(f"  Platform:           {sample_run.environment_info.get('platform')}")
            print(f"  CPU Architecture:   {sample_run.environment_info.get('cpu_architecture')}")
else:
    print("Provenance inspection skipped (experiment was not executed due to absent raw data).")

## 8. Summary Results Table and Client / Holdout Evaluation Metrics

Display the tabular summary results across models, regimes, participating clients, and the strict zero-shot held-out client (`heldout_model1`).

In [ ]:
if report is not None:
    print("=" * 120)
    print("Phase 3 Azure Baselines — Summary Results Table")
    print("=" * 120)
    print(format_summary_table(report.summary_table))
    
    print("\nDetailed Breakdown per Run:")
    for idx, run in enumerate(report.results):
        print(f"\n[{idx+1}/{len(report.results)}] {run.model_name.upper()} | Regime: {run.regime} | Client: {run.client_name} | Seed: {run.seed}")
        print(f"  Validation F1:            {run.val_f1_score:.4f} (at threshold {run.threshold:.4f})")
        print(f"  Best Epoch:               {run.best_epoch} (val score: {run.best_score})")
        print(f"  Test Windows (Total):     {run.total_windows_test} (pos: {run.window_positives_test}, neg: {run.window_negatives_test})")
        print(f"  Test Failure Episodes:    {run.distinct_failure_episodes_test} (eligible: {run.eligible_failure_episodes_test}, detected: {run.detected_failure_episodes_test})")
        
        # Window metrics
        twm = run.test_window_metrics
        ap_str = f"{twm.get('average_precision'):.4f}" if twm.get('average_precision') is not None else "N/A"
        roc_str = f"{twm.get('roc_auc'):.4f}" if twm.get('roc_auc') is not None else "N/A"
        print(f"  Window Metrics (Test):    AP={ap_str}, ROC-AUC={roc_str}, F1={twm.get('f1', 0.0):.4f}, Precision={twm.get('precision', 0.0):.4f}, Recall={twm.get('recall', 0.0):.4f}")
        
        # Event metrics
        tem = run.test_event_metrics
        rec_str = f"{tem.get('event_recall'):.4f}" if tem.get('event_recall') is not None else "N/A"
        far_str = f"{tem.get('false_alerts_per_100_machine_days'):.2f}" if tem.get('false_alerts_per_100_machine_days') is not None else "N/A"
        lead_str = f"{tem.get('mean_lead_hours'):.1f}h" if tem.get('mean_lead_hours') is not None else "N/A"
        print(f"  Event Metrics (Test):     Event Recall={rec_str}, FAR/100 machine-days={far_str}, Mean Lead Time={lead_str}")
        print(f"                            Alerts: total={tem.get('total_alerts')}, true={tem.get('true_alerts')}, false={tem.get('false_alerts')}")
        
        # Zero-shot heldout model1 metrics
        hwm = run.zero_shot_heldout_window_metrics
        hem = run.zero_shot_heldout_event_metrics
        h_ap_str = f"{hwm.get('average_precision'):.4f}" if hwm.get('average_precision') is not None else "N/A"
        h_rec_str = f"{hem.get('event_recall'):.4f}" if hem.get('event_recall') is not None else "N/A"
        h_far_str = f"{hem.get('false_alerts_per_100_machine_days'):.2f}" if hem.get('false_alerts_per_100_machine_days') is not None else "N/A"
        print(f"  Zero-Shot Holdout Model1: Test AP={h_ap_str}, Test F1={hwm.get('f1', 0.0):.4f}, Event Recall={h_rec_str}, FAR/100d={h_far_str}")
        print(f"  Resource Profiling:       Elapsed: {run.total_elapsed_seconds:.2f}s, Peak Mem: {run.peak_memory_mb or 0.0:.1f} MB, RSS: {run.process_rss_mb or 0.0:.1f} MB")
else:
    print("Results table skipped (experiment was not executed due to absent raw data).")

## 9. Machine-Cluster Bootstrap Uncertainty Estimation

Inspect the 95% percentile bootstrap confidence intervals ($[\text{CI}_{\text{lower}}, \text{CI}_{\text{upper}}]$) computed by resampling entire machines with replacement.

In [ ]:
if report is not None:
    print("=" * 90)
    print("Machine-Cluster Bootstrap Confidence Intervals (95% CI)")
    print("=" * 90)
    for run in report.results:
        print(f"\nRun: {run.model_name.upper()} | {run.regime} | {run.client_name} (Seed {run.seed})")
        print("  Window Metrics 95% CIs:")
        for metric_name in ["average_precision", "roc_auc", "f1", "precision", "recall"]:
            ci = run.test_window_ci.get(metric_name)
            if ci:
                pt = f"{ci['point_estimate']:.4f}" if ci.get('point_estimate') is not None else "N/A"
                lo = f"{ci['ci_lower']:.4f}" if ci.get('ci_lower') is not None else "N/A"
                hi = f"{ci['ci_upper']:.4f}" if ci.get('ci_upper') is not None else "N/A"
                print(f"    - {metric_name:<20s}: {pt} [95% CI: {lo}, {hi}] (valid: {ci['num_valid_draws']}/{ci['num_draws']})")
        
        print("  Event Metrics 95% CIs:")
        for metric_name in ["event_recall", "false_alerts_per_100_machine_days", "mean_lead_hours"]:
            ci = run.test_event_ci.get(metric_name)
            if ci:
                pt = f"{ci['point_estimate']:.4f}" if ci.get('point_estimate') is not None else "N/A"
                lo = f"{ci['ci_lower']:.4f}" if ci.get('ci_lower') is not None else "N/A"
                hi = f"{ci['ci_upper']:.4f}" if ci.get('ci_upper') is not None else "N/A"
                print(f"    - {metric_name:<34s}: {pt} [95% CI: {lo}, {hi}] (valid: {ci['num_valid_draws']}/{ci['num_draws']})")
else:
    print("Bootstrap inspection skipped (experiment was not executed due to absent raw data).")

## 10. Per-Seed Multi-Run Comparison & Compute Matching

When executed in `full` mode across seeds $[17, 23, 41]$, inspect variance across seeds and cumulative optimizer steps/samples processed for compute matching with downstream federated learning.

In [ ]:
if report is not None:
    from collections import defaultdict
    
    print("=" * 100)
    print("Per-Seed Multi-Run Variance & Compute Telemetry")
    print("=" * 100)
    
    grouped_runs = defaultdict(list)
    for run in report.results:
        key = (run.model_name, run.regime, run.client_name)
        grouped_runs[key].append(run)
        
    for (model_name, regime, client_name), runs in grouped_runs.items():
        print(f"\nTarget: {model_name.upper()} | {regime} | {client_name} ({len(runs)} seeds):")
        for r in runs:
            twm = r.test_window_metrics
            tem = r.test_event_metrics
            hwm = r.zero_shot_heldout_window_metrics
            ap = f"{twm.get('average_precision', 0.0):.4f}" if twm.get('average_precision') is not None else "N/A"
            f1 = f"{twm.get('f1', 0.0):.4f}"
            rec = f"{tem.get('event_recall', 0.0):.4f}" if tem.get('event_recall') is not None else "N/A"
            far = f"{tem.get('false_alerts_per_100_machine_days', 0.0):.2f}" if tem.get('false_alerts_per_100_machine_days') is not None else "N/A"
            h_ap = f"{hwm.get('average_precision', 0.0):.4f}" if hwm.get('average_precision') is not None else "N/A"
            print(
                f"  Seed {r.seed:2d}: Test AP={ap:<8} Test F1={f1:<8} Event Rec={rec:<8} FAR={far:<8} "
                f"Holdout AP={h_ap:<8} Steps={r.total_optimizer_steps:<6} Samples={r.total_samples_processed:<8} "
                f"Time={r.total_elapsed_seconds:6.1f}s PeakMem={r.peak_memory_mb or 0.0:5.1f}MB"
            )
else:
    print("Per-seed analysis skipped (experiment was not executed due to absent raw data).")

## 11. Methodological Boundaries, Research Assumptions & Limitations

1. **Window-Level vs. Event-Level Metrics:**
   - Consecutive hourly prediction windows for the same machine share 47 out of 48 observation hours.
   - A single failure episode generates up to 24 consecutive positive hourly windows. **Positive windows are NOT independent failure events.**
   - Window Average Precision (AP) measures binary window discrimination, but event-level operational metrics (Event Recall, False Alerts per 100 machine-days with a 24h cooldown) are the true measure of predictive maintenance utility.

2. **Centralized Regime as an Un-Privatized Reference:**
   - Centralized models are trained by pooling raw participant data into a single global training partition.
   - This provides a performance upper benchmark for non-private centralized aggregation; it does **not** satisfy federated privacy constraints.

3. **Strict Zero-Shot Generalization on Held-Out `model1`:**
   - `heldout_model1` is strictly excluded from all training updates, validation checks, hyperparameter tuning, and threshold selection.
   - Zero-shot evaluation assesses whether representations learned from `model2`, `model3`, and `model4` generalize to an unseen machine type without fine-tuning.

4. **Plumbing Mode vs. Scientific Verification:**
   - Smoke mode (`--mode smoke`) uses bounded stratified subsets ($\le 60$ samples), 2 epochs, 1 seed, and 30 bootstrap draws. It validates pipeline plumbing and execution paths only; its metrics must not be interpreted as model performance.
   - Full evaluation (`--mode full`) with seeds `[17, 23, 41]`, 25 epochs, and 1000 bootstrap draws is required for scientific claims.

5. **Data-Free Staging Integrity:**
   - In compliance with project rules, no raw datasets or fabricated outputs are embedded in this notebook.
   - Full execution and verification on real Azure CSVs will be conducted by Codex in the feature environment.